# BLOCK-T465 - Shutter timing linearity study

[BLOCK-T465]: https://rubinobs.atlassian.net/projects/BLOCK?selectedItem=com.atlassian.plugins.atlassian-connect-plugin:com.kanoah.test-manager__main-project-page#!/v2/testCase/BLOCK-T465

In [ ]:
%load_ext autoreload
%autoreload 2
import warnings
import numpy as np
import os

from lsst.ts.block.utils import build_configuration_schema
from lsst.ts.observing import ObservingBlock, ObservingScript

In [ ]:
name = "BLOCK-T465"
program = "BLOCK-T465"
reason = "shutter_timing_linearity"
constraints = []
scripts = []

try:
    output_folder = (
        os.environ["TS_CONFIG_OCS_DIR"] + "/Scheduler/observing_blocks_maintel"
    )
except KeyError:
    warnings.warn(
        "The environment variable 'TS_CONFIG_OCS_DIR' is not set. Using default folder 'output_blocks'."
    )
    output_folder = "output_blocks"

Define the configurable properties that we will use in the configuration schema

In [ ]:
# Defining configurable properties
properties = {
    "ra": {
        "description": "ICRS right ascension (hour) as a decimal number",
        "type": "number",
        "default": 18.9176944,
    },
    "dec": {
        "description": "ICRS declination (hour) as a decimal number",
        "type": "number",
        "default": -30.4832973,
    },
    "ignore": {
        "description": "List of CSCs to be ignored",
        "type": "array",
        "default": []
    }
}
    
block_number = name.split("-")[-1]
configuration_schema = build_configuration_schema(block_number, properties)
print(configuration_schema)

In [ ]:
# Define the ObservingBlock
track_target = ObservingScript(
    name="maintel/track_target.py",
    standard=True,
    parameters=dict(
        slew_icrs=dict(
            ra="$ra",
            dec="$dec",
            ignore="$ignore",    
        )
    )
)

scripts = []
for i in range(2, 31, 2):
    take_image_lsstcam =  ObservingScript(
    name="maintel/take_image_lsstcam.py",
    standard=True,
    parameters={
        "exp_times": i,
        "nimages": 2, 
        "image_type": "ENGTEST", 
        "reason": "BLOCK-T465", 
        "program": "BLOCK-T465", 
        "note": "Shutter_timing_test"
        }
    )
    scripts.append(take_image_lsstcam)

In [ ]:
block = ObservingBlock(
    name=name,
    program=program,
    configuration_schema=configuration_schema,
    scripts=scripts,
)

In [ ]:
block.model_dump_json(indent=2)

os.makedirs(output_folder, exist_ok=True)
output_path = f"{output_folder}/{name}.json"

with open(output_path, "w") as file:
    file.write(block.model_dump_json(indent=2))